# 01. The drone example (Fig. 48)

This is the canonical MCDP from Censi (2015), Fig. 48: a battery powers an actuator that has to lift the battery plus an extra payload. Because the battery's own mass appears in the lift it has to produce, the design problem is intrinsically recursive and only closes once the **Kleene fixed-point iteration** converges.

The model is built here as a single `FunctionDP` wrapped in a `Loop` on `battery_mass`. Compare with notebook **07** for the same model written modularly with the `System` builder.


## Imports

In [1]:
import math
from codesign import (
    Antichain, FunctionDP, Loop, Ports, Reals, solve,
)

## Physical constants

These follow Fig. 48 of the paper: a Li-ion battery (1.8 MJ/kg), gravity, and a quadratic drag coefficient for the actuator (10 W per N² of lift).

In [2]:
ALPHA = 1.8e6      # Li-ion specific energy, J/kg
G = 9.81           # gravity, m/s^2
C_LIFT = 10.0      # actuator coefficient, W per N^2 of lift

## The inner design problem

The functionality is `(endurance, extra_payload, extra_power, battery_mass)` and the resource is `(battery_mass, report_mass)`. Note that `battery_mass` appears on *both* sides: the inner DP receives the current iterate as a functionality input and emits a tightened estimate as a resource output. The `report_mass` is a mirrored copy of the same value so the outer R retains visibility of it (the `Loop` operator projects out the loop axis).


In [3]:
# Outer F has the mission spec plus the loop axis as an input.
F = Ports({
    "endurance":     Reals(unit="s"),
    "extra_payload": Reals(unit="kg"),
    "extra_power":   Reals(unit="W"),
    "battery_mass":  Reals(unit="kg"),    # current iterate, fed back in
})
# Inner R emits the tightened battery_mass plus a "report" mirror.
R = Ports({
    "battery_mass": Reals(unit="kg"),     # what Loop closes on
    "report_mass":  Reals(unit="kg"),     # what the outer world sees
})

def h(f):
    # Short-circuit: if any input has already diverged to +inf, propagate it.
    # This keeps the iteration well-defined when an intermediate iterate
    # exceeds the divergence cap and starts producing infinities.
    if (f["battery_mass"] == math.inf or
        f["endurance"] == math.inf or
        f["extra_payload"] == math.inf or
        f["extra_power"] == math.inf):
        return Antichain.singleton(R, {
            "battery_mass": math.inf, "report_mass": math.inf,
        })
    # Physics:
    #   lift force      = (battery + payload) * g
    #   actuator power  = C_LIFT * lift^2
    #   total power     = actuator + avionics extra
    #   energy required = total_power * endurance
    #   battery mass    = energy / specific_energy
    lift = (f["battery_mass"] + f["extra_payload"]) * G
    actuator_power = C_LIFT * lift * lift
    total_power = actuator_power + f["extra_power"]
    energy = total_power * f["endurance"]
    mass = energy / ALPHA
    # Both R components carry the same number; report_mass is the outer view.
    return Antichain.singleton(R, {
        "battery_mass": mass, "report_mass": mass,
    })

inner = FunctionDP(F=F, R=R, h_fn=h, name="drone")
# Close the loop on battery_mass: solve() will run the Kleene iteration
# until the iterate settles (or diverges to +inf).
drone = Loop(inner, axis="battery_mass")
drone

DP(loop(drone, axis=battery_mass): endurance:R+[s]×extra_payload:R+[kg]×extra_power:R+[W] -> A[report_mass:R+[kg]])

## Solving for several mission profiles

We sweep over short, medium, longer, marginal, and clearly-infeasible missions. The marginal and infeasible cases are correctly flagged: the loop axis is driven to `⊤` (infinity) when the recursion does not close on a finite battery mass.


In [4]:
# Five mission profiles, ordered roughly by difficulty.
cases = [
    ("Short, light",   dict(endurance=60.0,   extra_payload=0.10, extra_power=1.0)),
    ("Medium, modest", dict(endurance=300.0,  extra_payload=0.50, extra_power=5.0)),
    ("Longer mission", dict(endurance=600.0,  extra_payload=0.50, extra_power=5.0)),
    ("Marginal",       dict(endurance=600.0,  extra_payload=1.00, extra_power=10.0)),
    ("Infeasible",     dict(endurance=1800.0, extra_payload=1.00, extra_power=10.0)),
]
for label, f in cases:
    # Each solve runs an independent Kleene iteration. max_iter caps the
    # number of fixed-point steps; the feasible cases converge in ~10-25.
    result = solve(drone, f, max_iter=200)
    print(f"{label:<16} iters={result.iterations:>3}  "
          f"feasible={result.feasible}  {result.antichain}")

Short, light     iters=  9  feasible=True  Antichain[(report_mass=0.0003564 kg)]
Medium, modest   iters= 22  feasible=True  Antichain[(report_mass=0.04921 kg)]
Longer mission   iters= 41  feasible=True  Antichain[(report_mass=0.1283 kg)]
Marginal         iters= 16  feasible=False  Antichain[(report_mass=⊤)]
Infeasible       iters=  8  feasible=False  Antichain[(report_mass=⊤)]


## Reading the result

For the feasible cases, the antichain converges to a single battery mass that grows roughly with mission energy. The marginal and infeasible cases hit the divergence cap: the actuator can't lift a battery large enough to satisfy its own energy demand, so the Kleene ascent walks the loop axis to `⊤` and the solver reports `feasible=False`.

In notebook **07** we'll rebuild the same problem with the `System` builder, where battery and actuator are *independent* subsystems wired together with constraint equations.
